# Chat Template


Original Tutorial is [here](https://huggingface.co/docs/transformers/main/chat_templating?template=Zephyr).


### Introduction


The chat basics guide covers how to store chat histories and generate text from chat models using TextGenerationPipeline.

This guide is intended for more advanced users, and covers the underlying classes and methods, as well as the key concepts for understanding what’s actually going on when you chat with a model.

The critical insight needed to understand chat models is this: All causal LMs, whether chat-trained or not, continue a sequence of tokens. When causal LMs are trained, the training usually begins with “pre-training” on a huge corpus of text, which creates a “base” model. These base models are then often “fine-tuned” for chat, which means training them on data that is formatted as a sequence of messages. The chat is still just a sequence of tokens, though! The list of role and content dictionaries that you pass to a chat model get converted to a token sequence, often with control tokens like <|user|> or <|assistant|> or <|end_of_message|>, which allow the model to see the chat structure. There are many possible chat formats, and different models may use different formats or control tokens, even if they were fine-tuned from the same base model!

Don’t panic, though - you don’t need to memorize every possible chat format in order to use chat models. Chat models come with chat templates, which indicate how they expect chats to be formatted. You can access these with the apply_chat_template method. Let’s see two examples. Both of these models are fine-tuned from the same Mistral-7B base model:


#### Mistral


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
chat = [
    {"role": "user", "content": "Hello, how are you?"},
    {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
    {"role": "user", "content": "I'd like to show off how chat templating works!"},
]

tokenizer.apply_chat_template(chat, tokenize=False)

#### Zephyr


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceH4/zephyr-7b-beta")
chat = [
    {"role": "user", "content": "Hello, how are you?"},
    {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
    {"role": "user", "content": "I'd like to show off how chat templating works!"},
]

tokenizer.apply_chat_template(chat, tokenize=False)

/home/naoki/anaconda3/envs/g-retriever/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


"<|user|>\nHello, how are you?</s>\n<|assistant|>\nI'm doing great. How can I help you today?</s>\n<|user|>\nI'd like to show off how chat templating works!</s>\n"

Mistral-7B-Instruct uses [INST] and [/INST] tokens to indicate the start and end of user messages, while Zephyr-7B uses <|user|> and <|assistant|> tokens to indicate speaker roles. This is why chat templates are important - with the wrong control tokens, these models would have drastically worse performance.


### Using `apply_chat_template`


The input to apply_chat_template should be structured as a list of dictionaries with role and content keys. The role key specifies the speaker, and the content key contains the message. The common roles are:

- `user` for messages from the user
- `assistant` for messages from the model
- `system` for directives on how the model should act (usually placed at the beginning of the chat)

`apply_chat_template` takes this list and returns a formatted sequence. Set `tokenize=True` if you want to tokenize the sequence.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceH4/zephyr-7b-beta")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceH4/zephyr-7b-beta", device_map="auto")

messages = [
    {
        "role": "system",
        "content": "You are a friendly chatbot who always responds in the style of a pirate",
    },
    {
        "role": "user",
        "content": "How many helicopters can a human eat in one sitting?",
    },
]
tokenized_chat = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
)
print(tokenizer.decode(tokenized_chat[0]))

Loading checkpoint shards: 100%|██████████| 8/8 [00:07<00:00,  1.04it/s]


<|system|>
You are a friendly chatbot who always responds in the style of a pirate</s> 
<|user|>
How many helicopters can a human eat in one sitting?</s> 
<|assistant|>



Pass the tokenized chat to generate() to generate a response.


In [ ]:
outputs = model.generate(tokenized_chat, max_new_tokens=128)
print(tokenizer.decode(outputs[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/home/naoki/anaconda3/envs/g-retriever/lib/python3.12/site-packages/transformers/generation/utils.py:2489: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


<|system|>
You are a friendly chatbot who always responds in the style of a pirate</s> 
<|user|>
How many helicopters can a human eat in one sitting?</s> 
<|assistant|>
Matey, I'm afraid I must inform ye that humans cannot eat helicopters. Helicopters are not food, they are flying machines. Food is meant to be eaten, like a hearty plate o' grog, a savory bowl o' stew, or a delicious loaf o' bread. But helicopters, they be for transportin' and movin' around, not for eatin'. So, I'd say none, me hearties. None at all.</s>


#### Warning

Some tokenizers add special <bos> and <eos> tokens. Chat templates should already include all the necessary special tokens, and adding additional special tokens is often incorrect or duplicated, hurting model performance. When you format text with apply_chat_template(tokenize=False), make sure you set add_special_tokens=False if you tokenize later to avoid duplicating these tokens. This isn’t an issue if you use apply_chat_template(tokenize=True), which means it’s usually the safer option!


### `add_generation_prompt`


You may have noticed the add_generation_prompt argument in the above examples. This argument adds tokens to the end of the chat that indicate the start of an assistant response. Remember: Beneath all the chat abstractions, chat models are still just language models that continue a sequence of tokens! If you include tokens that tell it that it’s now in an assistant response, it will correctly write a response, but if you don’t include these tokens, the model may get confused and do something strange, like continuing the user’s message instead of replying to it!

Let’s see an example to understand what add_generation_prompt is actually doing. First, let’s format a chat without add_generation_prompt:


In [ ]:
tokenized_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print(tokenized_chat)

<|system|>
You are a friendly chatbot who always responds in the style of a pirate</s>
<|user|>
How many helicopters can a human eat in one sitting?</s>



Now, let’s format the same chat with `add_generation_prompt=True`:


In [ ]:
tokenized_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(tokenized_chat)

<|system|>
You are a friendly chatbot who always responds in the style of a pirate</s>
<|user|>
How many helicopters can a human eat in one sitting?</s>
<|assistant|>



When add_generation_prompt=True, <|im_start|>assistant is added at the end to indicate the start of an assistant message. This lets the model know an assistant response is next.

Not all models require generation prompts, and some models, like Llama, don’t have any special tokens before the assistant response. In these cases, add_generation_prompt has no effect.


### `continue_final_message`


The continue_final_message parameter controls whether the final message in the chat should be continued or not instead of starting a new one. It removes end of sequence tokens so that the model continues generation from the final message.

This is useful for “prefilling” a model response. In the example below, the model generates text that continues the JSON string rather than starting a new message. It can be very useful for improving the accuracy of instruction following when you know how to start its replies.


In [ ]:
chat = [
    {"role": "user", "content": "Can you format the answer in JSON?"},
    {"role": "assistant", "content": '{"name": "'},
]

formatted_chat = tokenizer.apply_chat_template(chat, tokenize=True, return_dict=True, continue_final_message=True)
model.generate(**formatted_chat)

AttributeError: 'list' object has no attribute 'shape'

### Model training


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceH4/zephyr-7b-beta")

chat1 = [
    {"role": "user", "content": "Which is bigger, the moon or the sun?"},
    {"role": "assistant", "content": "The sun."},
]
chat2 = [
    {"role": "user", "content": "Which is bigger, a virus or a bacterium?"},
    {"role": "assistant", "content": "A bacterium."},
]

dataset = Dataset.from_dict({"chat": [chat1, chat2]})
dataset = dataset.map(
    lambda x: {"formatted_chat": tokenizer.apply_chat_template(x["chat"], tokenize=False, add_generation_prompt=False)}
)
print(dataset["formatted_chat"])

Map: 100%|██████████| 2/2 [00:00<00:00, 520.03 examples/s]

Column(['<|user|>\nWhich is bigger, the moon or the sun?</s>\n<|assistant|>\nThe sun.</s>\n', '<|user|>\nWhich is bigger, a virus or a bacterium?</s>\n<|assistant|>\nA bacterium.</s>\n'])
